# 4 · Control flow

Two control-flow primitives:

1. **Conditional branching** via the `SKIPPED` sentinel — a node outputs `SKIPPED` on branches that shouldn't run. Skip propagates: any downstream node whose inputs are all `SKIPPED` is skipped too.
2. **For-each loops** via a compound node — a region of nodes between `for-each-start` and `for-each-end` that runs once per item.

In [ ]:
from typing import Annotated

from conductor import GraphEdge, GraphNode, NodeRegistry, compile
from conductor._sentinel import SKIPPED
from conductor.compound.for_each import FOR_EACH
from conductor.execution.engine import collect, execute
from conductor.types import NodeCategory
from conductor.widgets import ConnectionList, Output, Text

registry = NodeRegistry()


@registry.node("echo", version=1, name="Echo", description="Echoes input")
def echo(text: Annotated[str, Text(label="Input")]) -> Annotated[str, Output(label="Output")]:
    return text


@registry.node("upper", version=1, name="Upper", description="Uppercases")
def upper(text: Annotated[str, Text(label="Input")]) -> Annotated[str, Output(label="Output")]:
    return text.upper()


@registry.node("prefix", version=1, name="Prefix", description="Adds prefix")
def prefix(text: Annotated[str, Text(label="Input")]) -> Annotated[str, Output(label="Output")]:
    return f">> {text}" 

## Conditional node

Returns a tuple of outputs. The branch that should be **inactive** yields the `SKIPPED` sentinel.

In [ ]:
@registry.node(
    "if-empty",
    version=1,
    name="If Empty",
    description="Routes based on whether input is empty",
    category=NodeCategory.CONTROL,
)
def if_empty(
    text: Annotated[str, Text(label="Input")],
) -> tuple[
    Annotated[str, Output(label="Not empty")],
    Annotated[str, Output(label="Empty")],
]:
    if text.strip():
        return (text, SKIPPED)       # "Not empty" branch active
    else:
        return (SKIPPED, "empty")    # "Empty" branch active

### Wire it up

```text
  echo("hello") ──> if-empty ──(not empty)──> upper
                           └──(empty)──────> prefix   (skipped)
```

Because `"hello"` is non-empty, the `upper` node runs and `prefix` is skipped (its only input is `SKIPPED`).

In [ ]:
nodes = [
    GraphNode("text", "echo@1", {"text": "hello"}),
    GraphNode("cond", "if-empty@1", None),
    GraphNode("up", "upper@1", None),
    GraphNode("pf", "prefix@1", None),
]
edges = [
    GraphEdge("e1", "text", "cond", "result", "text"),
    GraphEdge("e2", "cond", "up", "output_1", "text"),   # "not empty"
    GraphEdge("e3", "cond", "pf", "output_2", "text"),   # "empty"
]

compiled = compile(nodes=nodes, edges=edges, registry=registry)
results = await collect(execute(compiled))

for nid, res in results.items():
    print(f"  {nid}: {res}")

## For-each loop

The engine ships with a `FOR_EACH` compound node type. Register two marker nodes in your registry — `for-each-start` (explodes a list into items) and `for-each-end` (collects results) — and pass `compound_types=[FOR_EACH]` to `compile()`.

The marker functions raise `NotImplementedError` — the compound node handler intercepts them.

In [ ]:
@registry.node(
    "for-each-start",
    version=1,
    name="For Each (Start)",
    description="Iterates over items",
    category=NodeCategory.CONTROL,
)
def for_each_start(
    items: Annotated[list[str], ConnectionList(label="Items")],
) -> tuple[
    Annotated[str, Output(label="Item")],
    Annotated[int, Output(label="Index")],
]:
    raise NotImplementedError("Handled by compound node")


@registry.node(
    "for-each-end",
    version=1,
    name="For Each (End)",
    description="Collects results",
    category=NodeCategory.CONTROL,
)
def for_each_end(
    item: Annotated[str, Text(label="Item")],
) -> Annotated[list[str], Output(label="Collected")]:
    raise NotImplementedError("Handled by compound node")

### Run a for-each loop

```text
  for-each-start(["alice", "bob", "charlie"])
      ─> upper (loop body, runs 3 times)
      ─> for-each-end  ──> ["ALICE", "BOB", "CHARLIE"]
```

In [ ]:
nodes_b = [
    GraphNode("start", "for-each-start@1", {"items": ["alice", "bob", "charlie"]}),
    GraphNode("body", "upper@1", None),
    GraphNode("end", "for-each-end@1", None),
]
edges_b = [
    GraphEdge("e1", "start", "body", "output_1", "text"),
    GraphEdge("e2", "body", "end", "result", "item"),
]

compiled_b = compile(
    nodes=nodes_b,
    edges=edges_b,
    registry=registry,
    compound_types=[FOR_EACH],
)
results_b = await collect(execute(compiled_b))

for nid, res in results_b.items():
    print(f"  {nid}: {res}")